**Stage 1: evaluate the GaitSTGCN checkpoint on a held-out test set (accuracy, precision, recall, F1, ROC-AUC, CIs, subgroups).**

Inputs: `stgcn_new_optimized.py` (upload it next to this notebook / to `/content` on Colab), the epoch-20 checkpoint of the 30k run (72.65% val accuracy), DenseGait `poses/`, `annotations_missing.csv` and `annotations.csv`.
Outputs (in `OUT_DIR`): `eval_metrics.json`, `eval_subgroups.csv`, `eval_predictions.csv` (per-clip; contains DenseGait labels, do not publish).

**Test set.** `file_check.ipynb` split DenseGait into the 175,247 clips that were on Google Drive (`annotations_present.csv`, from which the 30k training/validation subset was drawn) and 42,706 clips that never reached Drive (`annotations_missing.csv`). The missing clips were never seen in training or model selection, are spread evenly across the dataset, and have the same label balance (62.8% vs 62.7% female), so they serve as the held-out test set. DenseGait has no person IDs, so the same person appearing in another clip from the same source video cannot be ruled out.

Each clip is scored on its own (batch size 1). During training, clips were zero-padded to the longest clip in their batch and the model averages over padded frames, so batched scores depend on batch-mates.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # running locally

In [ ]:
import os, sys, json
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from sklearn import metrics as skm
from tqdm import tqdm
from stgcn_new_optimized import GaitSTGCN, GaitDatasetSTGCN, collate_fn_stgcn

BASE_DIR = os.environ.get('DENSEGAIT_DIR', '/content/drive/MyDrive/Gait Project/DenseGait')
EVAL_CSV   = os.path.join(BASE_DIR, 'annotations_missing.csv')   # held-out test set (never on Drive, never trained on)
FULL_ANNOTATIONS = os.path.join(BASE_DIR, 'annotations.csv')           # optional, for subgroup slices
POSES_DIR  = os.path.join(BASE_DIR, 'poses')
CHECKPOINT = os.environ.get('GAIT_CKPT', os.path.join(BASE_DIR, 'output', 'checkpoints', 'model_epoch_20.pth'))
OUT_DIR    = os.environ.get('EVAL_OUT', os.path.join(BASE_DIR, 'output', 'eval'))
SEED       = 0
N_BOOT     = 1000     # bootstrap resamples for 95% CIs
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
ann = pd.read_csv(EVAL_CSV)
print(f'Evaluating {len(ann):,} clips from {os.path.basename(EVAL_CSV)}')

# ── checkpoint sanity check: must be the 72.65% / 0.5422 epoch-20 model ──────
ckpt = torch.load(CHECKPOINT, map_location='cpu', weights_only=False)
print(f"checkpoint epoch={ckpt.get('epoch')} val_acc={ckpt.get('val_acc')} val_loss={ckpt.get('val_loss')}")
assert ckpt.get('epoch') == 20 and abs(ckpt['val_acc'] - 72.65) < 0.01 and abs(ckpt['val_loss'] - 0.5422) < 1e-4, \
    'Not the epoch-20 checkpoint of the 30k run that the thesis reports'

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
model = GaitSTGCN(in_channels=3, num_class=1, num_joints=18, edge_importance_weighting=True, dropout=0.3)
model.load_state_dict(ckpt['model_state_dict'])
model.to(device).eval()

# ── per-clip scores (batch size 1, same preprocessing + 300-frame cap as training) ──
# GaitDatasetSTGCN silently substitutes the next clip for an unreadable file; flag and drop those.
class Flagged(torch.utils.data.Dataset):
    def __init__(self, ds): self.ds = ds
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        x, y = self.ds[i]
        return x, y, i in self.ds.corrupted_indices

def collate_one(batch):
    x, y, bad = batch[0]
    poses, labels, _ = collate_fn_stgcn([(x, y)], max_seq_length=300)
    return poses, labels, bad

ds = Flagged(GaitDatasetSTGCN(EVAL_CSV, POSES_DIR, num_joints=18, num_coords=3, normalize=True, handle_missing=True))
loader = DataLoader(ds, batch_size=1, shuffle=False, collate_fn=collate_one,
                    num_workers=2 if sys.platform == 'linux' else 0)  # spawn-based workers can't pickle notebook classes
probs, labels, bad = [], [], []
with torch.no_grad():
    for poses, y, b in tqdm(loader):
        probs.append(torch.sigmoid(model(poses.to(device))).item())
        labels.append(int(y.item()))
        bad.append(b)
pred_df = pd.DataFrame({'file_name': ann['file_name'].values,
                        'female_score': ann['Female'].values,
                        'label': labels, 'prob_female': probs})
bad = np.array(bad)
print(f'Dropped {bad.sum()} unreadable clips')
pred_df = pred_df[~bad].reset_index(drop=True)
assert (pred_df['label'] == (pred_df['female_score'] > 0.5)).all(), 'file/label misalignment'
pred_df.to_csv(os.path.join(OUT_DIR, 'eval_predictions.csv'), index=False)

In [ ]:
# ── metrics (Female = 1 = positive class; threshold > 0.5 as in training) ────
def scores(y, p):
    y, p = np.asarray(y), np.asarray(p)
    yhat = (p > 0.5).astype(int)
    prec, rec, f1, sup = skm.precision_recall_fscore_support(y, yhat, labels=[0, 1], zero_division=0)
    out = {'n': int(len(y)), 'female_prevalence': float(y.mean()),
           'majority_baseline_acc': float(max(y.mean(), 1 - y.mean())),
           'accuracy': skm.accuracy_score(y, yhat), 'balanced_accuracy': skm.balanced_accuracy_score(y, yhat),
           'precision_male': prec[0], 'recall_male': rec[0], 'f1_male': f1[0],
           'precision_female': prec[1], 'recall_female': rec[1], 'f1_female': f1[1],
           'f1_macro': f1.mean(), 'confusion_matrix [[TN,FP],[FN,TP]]': skm.confusion_matrix(y, yhat, labels=[0, 1]).tolist()}
    if len(set(y)) == 2:
        out.update(roc_auc=skm.roc_auc_score(y, p), avg_precision=skm.average_precision_score(y, p),
                   log_loss=skm.log_loss(y, np.clip(p, 1e-7, 1 - 1e-7)))
    return {k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in out.items()}

def bootstrap_ci(y, p, keys=('accuracy', 'balanced_accuracy', 'f1_female', 'f1_male', 'f1_macro', 'roc_auc')):
    y, p = np.asarray(y), np.asarray(p)
    rng = np.random.default_rng(SEED)
    draws = [scores(y[i], p[i]) for i in (rng.integers(0, len(y), len(y)) for _ in range(N_BOOT))]
    return {k: [float(np.percentile([d[k] for d in draws if k in d], q)) for q in (2.5, 97.5)] for k in keys}

y, p = pred_df['label'].values, pred_df['prob_female'].values
result = {'eval_csv': os.path.basename(EVAL_CSV), 'checkpoint': os.path.basename(CHECKPOINT), 'all': scores(y, p), 'ci95': bootstrap_ci(y, p)}

# label-noise sensitivity: DenseGait 'Female' is an auto-generated soft score
conf = (pred_df['female_score'] <= 0.2) | (pred_df['female_score'] >= 0.8)
result['confident_labels_only'] = scores(y[conf], p[conf])

with open(os.path.join(OUT_DIR, 'eval_metrics.json'), 'w') as f:
    json.dump(result, f, indent=2)
print(json.dumps(result, indent=2))

In [ ]:
# ── subgroup slices on DenseGait's auto-generated attributes (not a fairness audit) ──
# Label balance differs a lot between slices, so compare balanced accuracy, not accuracy.
if os.path.exists(FULL_ANNOTATIONS):
    full = pd.read_csv(FULL_ANNOTATIONS)
    attrs = [c for c in full.select_dtypes('number').columns if c not in ('Female', 'file_number', 'track_id')]
    merged = pred_df.merge(full[['file_name'] + attrs], on='file_name', how='left')
    rows = []
    for a in attrs:
        s = merged[merged[a] > 0.5]
        if len(s) >= 200:
            r = scores(s['label'], s['prob_female'])
            lo, hi = bootstrap_ci(s['label'], s['prob_female'], keys=('balanced_accuracy',))['balanced_accuracy']
            rows.append({'attribute': a, 'n': r['n'], 'female_prevalence': r['female_prevalence'],
                         'accuracy': r['accuracy'], 'balanced_accuracy': r['balanced_accuracy'],
                         'balanced_accuracy_ci95_lo': lo, 'balanced_accuracy_ci95_hi': hi,
                         'recall_male': r['recall_male'], 'recall_female': r['recall_female'],
                         'f1_macro': r['f1_macro'], 'roc_auc': r.get('roc_auc')})
    sub = pd.DataFrame(rows).sort_values('balanced_accuracy')
    sub.to_csv(os.path.join(OUT_DIR, 'eval_subgroups.csv'), index=False)
    print(sub.to_string(index=False))
else:
    print('annotations.csv not found; skipping subgroup slices')